In [0]:
from pyspark.sql.types import  *
from pyspark.sql.functions import * 
from delta.tables import DeltaTable

In [0]:
df=spark.read.table('fmcg2.bronze.products')

In [0]:
display(df.limit(5))

In [0]:
df.printSchema()

drop duplicates

In [0]:
df.groupBy("product_id").count().where('count>1').display()



In [0]:
df=df.dropDuplicates(['product_id'])

In [0]:
df.select("category").distinct().display()

In [0]:
df=df.withColumn('category',initcap('category'))

correct spelling

In [0]:
df=df.withColumn('product_name',regexp_replace('product_name','protien','protein'))\
    .withColumn('category',regexp_replace('category','protien','protein'))

Create the column  of devison 

In [0]:

df = df.withColumn( "division", when(col("category") == "Energy Bars", "Nutrition Bars")
    .when(col("category") == "Protien Bars", "Protein Bars")
    .when(col("category") == "Granola & Cereals", "Breakfast Foods")
    .when(col("category") == "Recovery Dairy", "Dairy & Recovery")
    .when(col("category") == "Healthy Snacks", "Healthy Snacks")
    .when(col("category") == "Electrolyte Mix", "Hydration & Electrolytes")
    .otherwise("Other")
)

In [0]:
df.display()

In [0]:
df = df.withColumn(
    "variant",
    regexp_extract(col("product_name"), r"\((.*?)\)", 1)
)

In [0]:
df.display()

In [0]:
df=df.withColumnsRenamed({"product_id":"product_code",
                           "product_name":"product"})
    

In [0]:
display(df.limit(5))

In [0]:
df.write.format('delta')\
   .mode('overwrite')\
       .saveAsTable('fmcg2.silver.product_enr')


In [0]:
df.write.format('delta')\
    .mode('overwrite')\
        .saveAsTable('fmcg2.gold.product_gold')

In [0]:
from delta.tables import DeltaTable

In [0]:
target_table= DeltaTable.forName(spark,'fmcg.gold.dim_products')
child_table= spark.read.table('fmcg2.gold.product_gold')

target_table.alias('target').merge(child_table.alias('source'),'source.product_code= target.product_code')\
    .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
            .execute()